In [ ]:
pip install numpy pandas matplotlib scikit-learn ipykernel

In [ ]:
pip install jinja2

In [ ]:
# @title Bibliotecas
import numpy as np
import pandas as pd
import time
import sys
import math
import os

import matplotlib.pyplot as plt
import itertools
import string

from sklearn.ensemble import RandomForestRegressor
from IPython.display import display, HTML


In [ ]:
#@title Controle de Fluxo

QUERO_COLETAR_MAIS_DADOS = False
QUERO_RETREINAR_A_IA = False

AUTORIZAR_RADAR = True

SEED = 42
#np.random.seed(SEED)

VALOR_TREINO = 50

In [ ]:
# @title Classe TSPEnvironment

class TSPEnvironment:
  #Construtor da classe que recebe os valores de entrada da execução principal.
    def __init__(self, dados_tsplib):
        self.nome = dados_tsplib.get('NAME', 'TSP_Problem')
        self.n_cidades = dados_tsplib['DIMENSION']
        self.coordenadas = np.array(dados_tsplib['COORDS'])
        self.distancias = self._calcular_matriz_distancias()
        self.k_vizinhos = max(5, int(self.n_cidades * 0.20))
        self.lista_candidatos = self._gerar_radar()

  #MÉTODO PARA CALCULAR A DISTÂNCIA USANDO A FÓRMULA DESCRITA NO TRABALHO DO TSP
    def _calcular_matriz_distancias(self):
        n = self.n_cidades
        dist_matrix = np.zeros((n, n), dtype=int)
        for i in range(n):
            for j in range(i + 1, n):
                xd = self.coordenadas[i][0] - self.coordenadas[j][0]
                yd = self.coordenadas[i][1] - self.coordenadas[j][1]
                rij = math.sqrt(xd*xd + yd*yd)
                dij = int(math.floor(rij + 0.5))
                dist_matrix[i][j] = dist_matrix[j][i] = dij
        return dist_matrix

    #MÉTODO PARA GERAR O RADAR
    def _gerar_radar(self):
        return np.argsort(self.distancias, axis=1)[:, 1:self.k_vizinhos + 1]

    #ALGORITMO GULOSO
    def vizinho_mais_proximo(self):
          n = self.n_cidades
          visitados = np.zeros(n, dtype=bool)
          rota = [0]
          visitados[0] = True
          dist_total = 0
          atual = 0

          for _ in range(n - 1):
              dist_atuais = self.distancias[atual].astype(float)
              dist_atuais[visitados] = np.inf
              proximo = np.argmin(dist_atuais)
              rota.append(proximo)
              visitados[proximo] = True
              dist_total += self.distancias[atual][proximo]
              atual = proximo

          dist_total += self.distancias[rota[-1]][rota[0]]
          return dist_total

In [ ]:
# @title Classe Ant (Formiga)

class Ant:
    def __init__(self, n_cidades, alfa, beta):
        self.n_cidades = n_cidades
        self.alfa = alfa
        self.beta = beta
        self.rota = []
        self.distancia_total = 0

  
    def escolher_proxima(self, atual, nao_visitadas, matriz_feromonio, env):
        if AUTORIZAR_RADAR:
            usar_radar = self.n_cidades > 30
        else:
            usar_radar = False

        if usar_radar:
            candidatos = [c for c in env.lista_candidatos[atual] if c in nao_visitadas]
            if not candidatos: candidatos = list(nao_visitadas)
        else:
            candidatos = list(nao_visitadas)


        pesos = []
        for proxima in candidatos:
            f = matriz_feromonio[atual][proxima]
            d = 1.0 / (env.distancias[atual][proxima] + 1e-9)
            pesos.append((f ** self.alfa) * (d ** self.beta))


        soma = sum(pesos)
        probs = [p / soma for p in pesos]
        return np.random.choice(candidatos, p=probs)

    def percorrer_rota(self, env, matriz_feromonio):
        self.rota = [0]
        self.distancia_total = 0
        nao_visitadas = set(range(1, self.n_cidades))
        atual = 0

        while nao_visitadas:
            proxima = self.escolher_proxima(atual, nao_visitadas, matriz_feromonio, env)
            self.distancia_total += env.distancias[atual][proxima]
            self.rota.append(proxima)
            nao_visitadas.remove(proxima)
            atual = proxima
        self.distancia_total += env.distancias[self.rota[-1]][self.rota[0]]


In [ ]:
# @title Classe PheromoneManager (ControleFeromonio)

class PheromoneManager:
    def __init__(self, n_cidades, rho, Q):
        self.n_cidades = n_cidades
        self.rho = rho
        self.Q = Q
        self.matriz_feromonio = np.ones((n_cidades, n_cidades))

    def evaporar(self):
        self.matriz_feromonio *= (1.0 - self.rho)

    def depositar(self, lista_formigas):
        for formiga in lista_formigas:
            deposito = self.Q / formiga.distancia_total

            rota_arr = np.array(formiga.rota)

            origens = rota_arr[:-1]
            destinos = rota_arr[1:]

            self.matriz_feromonio[origens, destinos] += deposito
            self.matriz_feromonio[destinos, origens] += deposito

            u, v = rota_arr[-1], rota_arr[0]
            self.matriz_feromonio[u, v] += deposito
            self.matriz_feromonio[v, u] += deposito

In [ ]:
# @title Classe ACOTrainer

class ACOTrainer:
    def __init__(self, env, p_manager, n_formigas, n_iteracoes):
        self.env = env
        self.p_manager = p_manager
        self.n_formigas = n_formigas
        self.n_iteracoes = n_iteracoes
        self.melhor_distancia_global = float('inf')
        self.melhor_rota_global = None
        self.historico_distancias = []
        self.historico_feromonios = {}


    def treinar(self, alfa, beta, display_progress=True):
        formigas = [Ant(self.env.n_cidades, alfa, beta) for _ in range(self.n_formigas)]

        for i in range(self.n_iteracoes):
            for formiga in formigas:
                formiga.percorrer_rota(self.env, self.p_manager.matriz_feromonio)

                if formiga.distancia_total < self.melhor_distancia_global:
                    self.melhor_distancia_global = formiga.distancia_total
                    self.melhor_rota_global = formiga.rota.copy()

            self.p_manager.evaporar()
            self.p_manager.depositar(formigas)
            self.historico_distancias.append(self.melhor_distancia_global)
            self.historico_feromonios[i + 1] = (
                self.p_manager.matriz_feromonio.copy()
            )

            if display_progress and (i + 1) % 10 == 0:
                print(f"Iteração {i+1}/{self.n_iteracoes} | Melhor: {self.melhor_distancia_global:.2f}")

In [ ]:
# @title Função de Automação de Coleta

def automacao_coleta_dados(n_execucoes=30):
    nome_arquivo = os.path.join("..","data", "interim", "dados_para_random_forest.csv")
    os.makedirs(os.path.dirname(nome_arquivo), exist_ok=True)
    
    base_dados = []

    for i in range(n_execucoes):
        n_c = int(np.exp(np.random.uniform(np.log(20), np.log(500))))
        a_test = np.random.uniform(0.5, 2.5)
        b_test = np.random.uniform(1.5, 5.0)

        formigas_coleta = 30 if n_c > 200 else int(n_c * 0.5)

        coords = np.random.randint(0, 1000, size=(n_c, 2))

        dados_teste = {
            "NAME": f"TSP_{n_c}",
            "DIMENSION": n_c,
            "COORDS": coords
        }

        env_teste = TSPEnvironment(dados_teste)
        p_man = PheromoneManager(n_c, rho=0.1, Q=env_teste.vizinho_mais_proximo())
        trainer = ACOTrainer(env_teste, p_man, n_formigas=formigas_coleta, n_iteracoes=30)

        start = time.time()
        trainer.treinar(alfa=a_test, beta=b_test, display_progress=False)
        end = time.time()

        base_dados.append({
            "num_cidades": n_c,
            "alfa_utilizado": round(a_test, 2),
            "beta_utilizado": round(b_test, 2),
            "resultado_distancia": round(trainer.melhor_distancia_global, 2),
            "tempo_segundos": round(end - start, 4)
        })

        if (i+1) % 100 == 0:
            print(f"Processado: {i+1}/{n_execucoes} | Cidades: {n_c} | Tempo: {round(end-start, 2)}s")

    print("\n  Lote de processamento finalizado!")
    df_novos = pd.DataFrame(base_dados)

    if not os.path.isfile(nome_arquivo):
        df_novos.to_csv(nome_arquivo, index=False)
    else:
        df_novos.to_csv(nome_arquivo, mode='a', header=False, index=False)

    return df_novos

In [ ]:
# @title Tabela de Treino
if QUERO_COLETAR_MAIS_DADOS:
    tabela_treino = automacao_coleta_dados(VALOR_TREINO)

In [ ]:
# @title  Base de Dados Coletados

nome_arquivo = os.path.join("..","data", "interim", "dados_para_random_forest.csv")

if os.path.exists(nome_arquivo):
    df_exibicao = pd.read_csv(nome_arquivo).tail(10)

    # Estilização
    estilo_treino = df_exibicao.style.format({
        'alfa_utilizado': '{:.2f}',
        'beta_utilizado': '{:.2f}',
        'resultado_distancia': '{:.2f}',
        'tempo_segundos': '{:.2f}s'
    }).set_caption("  ÚLTIMAS EXPERIÊNCIAS REGISTRADAS NO DATASET") \
      .set_properties(**{
          'text-align': 'center',
          'color': '#ffffff',
          'border': '1px solid #444'
      }).set_table_styles([
          {'selector': 'caption', 'props': [('color', '#3498db'), ('font-size', '16px'), ('font-weight', 'bold'), ('padding', '10px')]},
          {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', '#3498db'), ('border', '1px solid #444'), ('text-align', 'center')]},
          {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#2c3e50')]},
          {'selector': 'tr:nth-child(odd)', 'props': [('background-color', '#34495e')]},
          {'selector': 'tr:hover', 'props': [('background-color', '#2980b9')]}
      ])

    display(estilo_treino)

    total_linhas = len(pd.read_csv(nome_arquivo))
    print(f"\n  Total de experiências no Dataset: {total_linhas}")
    if total_linhas < 1000:
        print(f"  Meta para ativar IA: {total_linhas}/1000")
    else:
        print("  Base de dados robusta! IA pronta para treinamento.")
else:
    print("Arquivo CSV ainda não foi criado. Rode a Célula de Coleta primeiro.")

In [ ]:
# @title Treinamento da Inteligência Artificial (Random Forest)

def treinar_especialista_ia():

    nome_arquivo = os.path.join("..","data", "interim", "dados_para_random_forest.csv")
    
    if not os.path.isfile(nome_arquivo):
        print("Arquivo de dados não encontrado! Continue coletando dados primeiro.")
        return None

    df = pd.read_csv(nome_arquivo)

    if len(df) < 5000:
        print(f" Dados insuficientes ({len(df)}/5000). Continue na fase de coleta aleatória.")
        return None

    np.random.seed(SEED)
    df['score'] = df['resultado_distancia'] / df['num_cidades']

    limiar = df['score'].quantile(0.30)
    dados_elite = df[df['score'] <= limiar].copy()

    print(f"  IA estudando os {len(dados_elite)} melhores exemplos de {len(df)} simulações...")

    X = dados_elite[['num_cidades']].values
    y = dados_elite[['alfa_utilizado', 'beta_utilizado']].values

    modelo = RandomForestRegressor(n_estimators=100, random_state=SEED)
    modelo.fit(X, y)

    print("  IA Treinada e pronta para dar palpites!")
    return modelo

if QUERO_RETREINAR_A_IA:
    modelo_ia = treinar_especialista_ia()
else:
    print("Está executando essa celula")
    modelo_ia = None

In [ ]:
# @title Geração e Salvamento da Tabela Geral da IA no Drive

def gerar_tabela_geral_ia(modelo_treinado):
    if modelo_treinado is None:
        print("Erro: O modelo de IA não está treinado. Execute a célula de treinamento primeiro.")
        return None

    caminho_saida_geral = os.path.join("..","data", "processed", "parametros_gerais_ia.csv")
    os.makedirs(os.path.dirname(caminho_saida_geral), exist_ok=True)


    tamanhos_gerais = list(range(20, 401))

    resultados_gerais = []

    print("IA gerando o mapeamento geral de parâmetros (de 20 a 400 cidades)...")

    for n_cidades in tamanhos_gerais:
        predicao = modelo_treinado.predict([[n_cidades]])

        alfa_ia = round(predicao[0][0], 2)
        beta_ia = round(predicao[0][1], 2)

        iteracoes = np.random.randint(50, 100)

        resultados_gerais.append({
            "dimensao": n_cidades,
            "alfa_ideal": alfa_ia,
            "beta_ideal": beta_ia,
            "n_iteracoes": iteracoes
        })

    df_geral_ia = pd.DataFrame(resultados_gerais)

    df_geral_ia.to_csv(caminho_saida_geral, index=False)

    print(f"\nTabela Geral da IA salva com {len(df_geral_ia)} cenários mapeados em:\n {caminho_saida_geral}")
    return df_geral_ia

df_parametros_gerais = gerar_tabela_geral_ia(modelo_ia)

In [ ]:
# @title Execução do Algoritmo (Modo Híbrido IA/Aleatório)

n_cidades_final = int(np.exp(np.random.uniform(np.log(20), np.log(500))))
caminho_tabela_ia = os.path.join("..","data", "processed", "parametros_gerais_ia.csv")

usei_ia = False

if os.path.exists(caminho_tabela_ia):

    with open(caminho_tabela_ia, mode='r', encoding='utf-8') as f:
        linhas = f.read().splitlines()


    for linha in linhas[1:]:
        if not linha.strip():
            continue
        coluna = linha.split(',')
        
        if int(coluna[0]) == n_cidades_final:
            alfa_exec = round(float(coluna[1]), 2)
            beta_exec = round(float(coluna[2]), 2)
            n_iteracoes_final = int(coluna[3]) 
            status_ia = "ATIVADA"
            usei_ia = True
            print(f"A IA foi usada (Correspondência Exata para {n_cidades_final} cidades)")
            break

if not usei_ia:
    alfa_exec = round(np.random.uniform(0.5, 2.0), 2)
    beta_exec = round(np.random.uniform(2.0, 5.0), 2)
    n_iteracoes_final = np.random.randint(50, 100)
    status_ia = "DESATIVADA"
    print("Isso foi executado (Fallback ativo)")

n_formigas_final = 30 if n_cidades_final > 200 else int(n_cidades_final * 0.5)



coords = np.random.randint(
    0,
    1000,
    size=(n_cidades_final, 2)
)

dados_teste = {
    "NAME": f"TSP_{n_cidades_final}",
    "DIMENSION": n_cidades_final,
    "COORDS": coords
}

env_final = TSPEnvironment(dados_teste)
p_man_final = PheromoneManager(n_cidades_final, rho=0.1, Q=env_final.vizinho_mais_proximo())
treinador_final = ACOTrainer(env_final, p_man_final, n_formigas_final, n_iteracoes_final)

start = time.time()
treinador_final.treinar(alfa=alfa_exec, beta=beta_exec)
end = time.time()


In [ ]:
# @title Painel de Resultados

if usei_ia:
    status_texto = "ATIVADA (Fase de Teste)"
    cor_status = "#27ae60" 
    cor_topo = "#27ae60"
else:
    status_texto = "DESLIGADA (Fase de Coleta)"
    cor_status = "#d35400" 
    cor_topo = "#34495e" 

html_painel = f"""
<div style="font-family: Arial, sans-serif; max-width: 800px; color: #1a1a1a;">

    <table style="width:100%; border-collapse: collapse; margin-bottom: 15px; border: 2px solid {cor_topo};">
        <tr style="background-color: {cor_topo}; color: white; text-align: center;">
            <th colspan="2" style="padding: 12px; font-size: 1.1em;">  CONFIGURAÇÃO DO EXPERIMENTO</th>
        </tr>
        <tr>
            <td style="padding: 10px; border: 1px solid #ccc; width: 45%; background-color: #f2f2f2; color: #333;"><b>Status da IA</b></td>
            <td style="padding: 10px; border: 1px solid #ccc; background-color: white; color: {cor_status}; font-weight: bold;">{status_texto}</td>
        </tr>
        <tr>
            <td style="padding: 10px; border: 1px solid #ccc; background-color: #f2f2f2; color: #333;"><b>Dimensão do Mapa</b></td>
            <td style="padding: 10px; border: 1px solid #ccc; background-color: white; color: #000;">{n_cidades_final} Cidades | {n_formigas_final} Formigas</td>
        </tr>
        <tr>
            <td style="padding: 10px; border: 1px solid #ccc; background-color: #f2f2f2; color: #333;"><b>Pesos Utilizados</b></td>
            <td style="padding: 10px; border: 1px solid #ccc; background-color: white; color: #000;">Alfa: {alfa_exec} | Beta: {beta_exec}</td>
        </tr>
    </table>

    <table style="width:100%; border-collapse: collapse; margin-bottom: 15px; border: 2px solid #2980b9;">
        <tr style="background-color: #2980b9; color: white; text-align: center;">
            <th colspan="2" style="padding: 12px; font-size: 1.1em;">  PROCESSAMENTO E PERFORMANCE</th>
        </tr>
        <tr>
            <td style="padding: 10px; border: 1px solid #ccc; width: 45%; background-color: #f2f2f2; color: #333;"><b>Tempo Total</b></td>
            <td style="padding: 10px; border: 1px solid #ccc; background-color: white; color: #000;">{end-start:.2f} segundos</td>
        </tr>
        <tr>
            <td style="padding: 10px; border: 1px solid #ccc; background-color: #f2f2f2; color: #333;"><b>Modo de Visão</b></td>
            <td style="padding: 10px; border: 1px solid #ccc; background-color: white; color: #000;">{'LIMITADA (RADAR 20%)' if n_cidades_final > 30 else 'TOTAL (100%)'}</td>
        </tr>
    </table>

    <table style="width:100%; border-collapse: collapse; border: 2px solid #16a085;">
        <tr style="background-color: #16a085; color: white; text-align: center;">
            <th colspan="2" style="padding: 12px; font-size: 1.1em;">  RESULTADO FINAL</th>
        </tr>
        <tr style="background-color: white;">
            <td style="padding: 15px; border: 1px solid #ccc; width: 45%; color: #333;"><b>Melhor Distância</b></td>
            <td style="padding: 15px; border: 1px solid #ccc; color: #16a085; font-size: 1.5em; font-weight: bold;">{treinador_final.melhor_distancia_global:.2f}</td>
        </tr>
    </table>
</div>
"""

display(HTML(html_painel))

In [ ]:
# @title Caminhos Gulosos (Início, Meio e Fim)

def calcular_baselines_triplo(env):
    pontos_partida = [0, env.n_cidades // 2, env.n_cidades - 1]
    resultados = {}

    for inicio in pontos_partida:
        visitados = np.zeros(env.n_cidades, dtype=bool)
        rota = [inicio]
        visitados[inicio] = True
        dist_total = 0.0
        atual = inicio

        for _ in range(env.n_cidades - 1):

              dist_atuais = env.distancias[atual].astype(float)
              dist_atuais[visitados] = np.inf

              proximo = np.argmin(dist_atuais)


              dist_total += env.distancias[atual][proximo]
              visitados[proximo] = True
              rota.append(proximo)
              atual = proximo

        dist_total += env.distancias[rota[-1]][rota[0]]
        resultados[inicio] = dist_total

    return resultados

dict_baselines = calcular_baselines_triplo(env_final)

print("**** RESULTADOS DOS BASELINES (GULOSO) ****\n")
for cidade, valor in dict_baselines.items():
    print(f" Partida Cidade {cidade: >3}: {valor:.2f}")

In [ ]:
# @title Mapa de Cidades

def plotar_universo_cru(env):
    coords = env.coordenadas
    n = env.n_cidades

    def gerar_nomes(n):
        nomes = []
        for i in range(1, 4):
            for combo in itertools.product(string.ascii_uppercase, repeat=i):
                nomes.append(''.join(combo))
                if len(nomes) == n:
                    return nomes
        return nomes

    letras = gerar_nomes(n)

    plt.figure(figsize=(14, 9))

    plt.scatter(coords[:, 0], coords[:, 1], color='black', s=60, zorder=5)

    for i, (x, y) in enumerate(coords):
        plt.text(x + 1.2, y + 1.2, letras[i], fontsize=9, fontweight='bold', color='darkblue')

    plt.title(f"Mapa Original: {n} Cidades (Posicionamento Geográfico)")
    plt.xlabel("Eixo X")
    plt.ylabel("Eixo Y")
    plt.grid(True, linestyle=':', alpha=0.6)

    plt.xlim(coords[:, 0].min() - 5, coords[:, 0].max() + 10)
    plt.ylim(coords[:, 1].min() - 5, coords[:, 1].max() + 10)

    plt.show()

plotar_universo_cru(env_final)

In [ ]:
# @title Visualização da Melhor Rota


def plotar_rota_final_universal(env, treinador):
    coords = env.coordenadas
    rota = treinador.melhor_rota_global
    n = env.n_cidades

    if rota is None:
        print("Sem rota para plotar.")
        return

    def gerar_nomes(n_cidades):
        nomes = []
        for i in range(1, 4):
            for combo in itertools.product(string.ascii_uppercase, repeat=i):
                nomes.append(''.join(combo))
                if len(nomes) == n_cidades: return nomes
        return nomes

    letras = gerar_nomes(n)

    plt.figure(figsize=(15, 10))

    ordenado = coords[rota]
    ordenado = np.vstack([ordenado, ordenado[0]])

    plt.plot(ordenado[:, 0], ordenado[:, 1], color='royalblue',
             linewidth=2, alpha=0.7, zorder=1, label='Melhor Rota')

    if n < 50:
        for i in range(len(rota)):
            p1, p2 = rota[i], rota[(i + 1) % len(rota)]
            mx, my = (coords[p1] + coords[p2]) / 2
            plt.text(mx, my, f"{env.distancias[p1][p2]:.1f}", color='darkgreen',
                     fontsize=7, fontweight='bold', ha='center', bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

    plt.scatter(coords[:, 0], coords[:, 1], color='red', s=60, zorder=5)

    if n < 100:
        centro_x, centro_y = np.mean(coords[:, 0]), np.mean(coords[:, 1])
        for i, (x, y) in enumerate(coords):
            ox = 2.0 if x > centro_x else -2.0
            oy = 2.0 if y > centro_y else -2.0
            plt.text(x + ox, y + oy, letras[i], fontsize=9, fontweight='bold', ha='center')

    plt.title(f"ACO: Rota Final Otimizada ({n} Cidades)\nDistância Total: {treinador.melhor_distancia_global:.2f} km")
    plt.grid(True, linestyle=':', alpha=0.5)

    plt.xlim(coords[:,0].min() - 15, coords[:,0].max() + 15)
    plt.ylim(coords[:,1].min() - 15, coords[:,1].max() + 15)

    plt.show()

plotar_rota_final_universal(env_final, treinador_final)

In [ ]:
# @title Tabela de Evolução do Conhecimento

def gerar_tabela_feromonios_detalhada(treinador):
    if not treinador or not treinador.historico_feromonios:
        print("Aviso: Nenhum histórico de feromônios encontrado. O treino rodou?")
        return pd.DataFrame()

    n = treinador.env.n_cidades

    def gerar_nomes(n_cidades):
        nomes = []
        for i in range(1, 4):
            for combo in itertools.product(string.ascii_uppercase, repeat=i):
                nomes.append(''.join(combo))
                if len(nomes) == n_cidades: return nomes
        return nomes

    letras = gerar_nomes(n)
    relatorio = []

    iteracoes = sorted(treinador.historico_feromonios.keys())

    for itera in iteracoes:
        matriz = treinador.historico_feromonios[itera]

        max_idx = np.unravel_index(np.nanargmax(matriz), matriz.shape)
        cidade_a, cidade_b = letras[max_idx[0]], letras[max_idx[1]]

        relatorio.append({
            "Rodada": itera,
            "Intensidade Máxima": np.max(matriz),
            "Aresta mais Forte": f"{cidade_a} ↔ {cidade_b}",
            "Média Global": np.mean(matriz),
            "Conhecimento Total": np.sum(matriz)
        })

    return pd.DataFrame(relatorio)

tabela_ajustada = gerar_tabela_feromonios_detalhada(treinador_final)

if not tabela_ajustada.empty:
    tabela_estilizada = tabela_ajustada.style.format({
        'Intensidade Máxima': '{:.2f}',
        'Média Global': '{:.4f}',
        'Conhecimento Total': '{:.1f}'
    }).set_properties(**{'text-align': 'center'}) \
      .set_table_styles([{
          'selector': 'th',
          'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center')]
      }])

    display(tabela_estilizada)

In [ ]:
# @title Desempenho: Inteligência Coletiva vs. Estratégias Gulosas

plt.figure(figsize=(12, 6))

if usei_ia:
    texto_legenda = 'Evolução ACO (Otimizado por IA)'
else:
    texto_legenda = 'Evolução ACO (Fase de Coleta)'

plt.plot(range(1, len(treinador_final.historico_distancias) + 1),
         treinador_final.historico_distancias,
         marker='o', markersize=4, color='forestgreen',
         label=texto_legenda, linewidth=2)

cores = ['red', 'darkorange', 'darkred']
for i, (cidade, valor) in enumerate(dict_baselines.items()):
    plt.axhline(y=valor, color=cores[i], linestyle='--', alpha=0.7,
                label=f'Guloso (Início Cidade {cidade})')

plt.title("Desempenho: Inteligência Coletiva vs. Estratégias Gulosas", fontsize=14)
plt.xlabel("Iteração (Geração da Colônia)", fontsize=12)
plt.ylabel("Distância Total", fontsize=12)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)

plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()